# SAGES CVS rubric annotation (frame-level)

In [ ]:
from __future__ import annotations

import csv
import json
import random
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from PIL import Image

In [ ]:
# -----------------------------
# CONFIG
# -----------------------------
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'rubrics/cvs_rubrics_v3.json').is_file())
FRAMES_ROOT = REPO_ROOT / "data" / "frames"
LABELS_ROOT = REPO_ROOT / "data" / "CVS_Challenge_SAGES_v1"
MANIFESTS_ROOT = REPO_ROOT / "data" / "manifests" / "cvs_challenge_sages_v1"
RUBRICS_PATH = REPO_ROOT / "rubrics" / "cvs_rubrics_v3.json"
ANNOTATIONS_DIR = REPO_ROOT / "annotations" / "new"

SEED = 13
BATCH_IDX = 0
N_PER_LABEL = 10          # 10 good + 10 bad per criterion = 60 total
N_FROM_TRAIN_REST = 5     # 5 out of 60 from train_rest
SHOW_IMAGES = True

print("FRAMES_ROOT:", FRAMES_ROOT)
print("LABELS_ROOT:", LABELS_ROOT)
print("RUBRICS_PATH:", RUBRICS_PATH)

In [ ]:
# -----------------------------
# Helpers
# -----------------------------

def iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def majority_vote(votes: List[int]) -> Optional[int]:
    if not votes:
        return None
    return 1 if sum(votes) >= 2 else 0


def load_frame_labels(video_id: str, split: str) -> Dict[int, Dict[str, int]]:
    """Load per-frame majority-vote labels for a video."""
    csv_path = LABELS_ROOT / split / "labels" / video_id / "frame.csv"
    if not csv_path.exists():
        return {}
    labels: Dict[int, Dict[str, int]] = {}
    with csv_path.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            fid = int(row["frame_id"])
            frame_labels = {}
            for c in ("c1", "c2", "c3"):
                votes = []
                for r in (1, 2, 3):
                    val = row.get(f"{c}_rater{r}")
                    if val is not None and val.strip() != "":
                        votes.append(int(val))
                mv = majority_vote(votes)
                if mv is not None:
                    frame_labels[c] = mv
            labels[fid] = frame_labels
    return labels


def load_manifest_frames(manifest_name: str) -> List[Dict[str, Any]]:
    """Load all frames from a manifest with their labels."""
    manifest_path = MANIFESTS_ROOT / f"{manifest_name}.jsonl"
    frames = []
    for rec in iter_jsonl(manifest_path):
        video_id = str(rec["id"])
        split = rec["split"]
        frame_ids = rec.get("frame_ids", [])
        labels_by_frame = load_frame_labels(video_id, split)
        for fid in frame_ids:
            fid_int = int(fid)
            fl = labels_by_frame.get(fid_int, {})
            if not fl or len(fl) < 3:
                continue
            img_path = FRAMES_ROOT / split / video_id / f"frame_{fid_int:06d}.jpg"
            frames.append({
                "video_id": video_id,
                "frame_id": fid_int,
                "split": split,
                "image_path": str(img_path.resolve()),
                "manifest": manifest_name,
                "gt": {"c1": fl["c1"], "c2": fl["c2"], "c3": fl["c3"]},
                "label_combo": (fl["c1"], fl["c2"], fl["c3"]),
            })
    return frames


def show_image(path: Path, title: str) -> None:
    if not path.exists():
        print(f"[WARN] Missing image: {path}")
        return
    with Image.open(path) as im:
        img = im.convert("RGB")
    plt.figure(figsize=(4, 3))
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
# -----------------------------
# Load rubrics
# -----------------------------
rubrics_data = json.loads(RUBRICS_PATH.read_text())
criteria_blocks = rubrics_data["criteria"]
criterion_order = ["C1", "C2", "C3"]

rubric_items: List[Dict[str, object]] = []
rubric_items_by_criterion: Dict[str, List[Dict[str, object]]] = {}
for criterion in criterion_order:
    block = criteria_blocks[criterion]
    items = list(block["items"])
    rubric_items_by_criterion[criterion] = items
    for item in items:
        rubric_items.append({
            "criterion": criterion,
            "id": item["id"],
            "text": item["text"],
            "weight": item["weight"],
            "label_type": item.get("label_type"),
        })

print("Rubric items:", len(rubric_items))
for criterion in criterion_order:
    print(criterion, "items:", len(rubric_items_by_criterion[criterion]))

In [ ]:
# -----------------------------
# Load frames from both manifests
# -----------------------------
print("Loading train_rest...")
frames_train_rest = load_manifest_frames("train_rest")
print(f"  {len(frames_train_rest)} frames from {len(set(f['video_id'] for f in frames_train_rest))} videos")

print("Loading train_dev...")
frames_train_dev = load_manifest_frames("train_dev")
print(f"  {len(frames_train_dev)} frames from {len(set(f['video_id'] for f in frames_train_dev))} videos")

In [ ]:
# -----------------------------
# Sampling strategy
# -----------------------------
# Goal: 60 frames total (10 good + 10 bad per criterion)
#   - 5 from train_rest with diverse label combos (c1,c2,c3)
#   - 55 from train_dev
#   - All from as many different videos as possible
#
# Each frame is assigned to exactly one (criterion, label) group.
# The 6 groups: (c1,1), (c1,0), (c2,1), (c2,0), (c3,1), (c3,0)

rng = random.Random(SEED)

group_specs: List[Tuple[str, int]] = [
    ("c1", 1), ("c1", 0),
    ("c2", 1), ("c2", 0),
    ("c3", 1), ("c3", 0),
]

# --- Step 1: Pick 5 from train_rest with diverse label combos ---
# Group train_rest frames by label combo
rest_by_combo: Dict[Tuple[int,int,int], List[Dict]] = defaultdict(list)
for f in frames_train_rest:
    rest_by_combo[f["label_combo"]].append(f)

print(f"Label combos available in train_rest: {len(rest_by_combo)}")
for combo, flist in sorted(rest_by_combo.items()):
    vids = set(f["video_id"] for f in flist)
    print(f"  {combo}: {len(flist)} frames, {len(vids)} videos")

# Pick 5 different combos, then 1 frame from each (different videos)
available_combos = list(rest_by_combo.keys())
rng.shuffle(available_combos)
selected_combos = available_combos[:N_FROM_TRAIN_REST]
# If fewer than 5 combos available, pick extra from the same combos
while len(selected_combos) < N_FROM_TRAIN_REST:
    selected_combos.append(rng.choice(available_combos))

train_rest_picks: List[Dict] = []
used_videos: set = set()
for combo in selected_combos:
    candidates = [f for f in rest_by_combo[combo] if f["video_id"] not in used_videos]
    if not candidates:
        candidates = rest_by_combo[combo]  # fallback
    pick = rng.choice(candidates)
    train_rest_picks.append(pick)
    used_videos.add(pick["video_id"])

print(f"\nSelected {len(train_rest_picks)} frames from train_rest:")
for p in train_rest_picks:
    print(f"  video={p['video_id'][:12]}... frame={p['frame_id']} combo={p['label_combo']}")

# Show the train_rest images
for p in train_rest_picks:
    img_path = Path(p["image_path"])
    title = f"{p['video_id'][:12]}... | frame {p['frame_id']} | combo={p['label_combo']}"
    show_image(img_path, title)

In [ ]:
# --- Step 2: Assign train_rest picks to groups ---
# Each frame can go into any group where it matches the criterion label.
# Spread them across different groups as much as possible.

group_needs = {(c, g): N_PER_LABEL for c, g in group_specs}  # each group needs 10
group_entries: Dict[Tuple[str,int], List[Dict]] = {k: [] for k in group_needs}

# For each train_rest pick, find which groups it qualifies for
# and assign it to one (prefer groups with fewer assignments)
assigned_groups = set()
for pick in train_rest_picks:
    gt = pick["gt"]
    eligible = []
    for (c, g) in group_specs:
        if gt[c] == g and (c, g) not in assigned_groups:
            eligible.append((c, g))
    if not eligible:
        # All eligible groups already have a train_rest pick; allow any
        eligible = [(c, g) for (c, g) in group_specs if gt[c] == g]
    chosen_group = rng.choice(eligible)
    assigned_groups.add(chosen_group)
    entry = {
        "frame": pick,
        "sample_group": {"criterion": chosen_group[0], "gt": chosen_group[1]},
        "source": "train_rest",
    }
    group_entries[chosen_group].append(entry)

print("Train_rest assignments:")
for (c, g), entries in group_entries.items():
    rest_count = sum(1 for e in entries if e["source"] == "train_rest")
    if rest_count:
        print(f"  ({c}, {g}): {rest_count} from train_rest")

In [ ]:
# --- Step 3: Fill remaining slots from train_dev, maximizing video diversity ---

# Index train_dev frames by (criterion, label)
dev_by_group: Dict[Tuple[str,int], List[Dict]] = defaultdict(list)
for f in frames_train_dev:
    gt = f["gt"]
    for c in ("c1", "c2", "c3"):
        dev_by_group[(c, gt[c])].append(f)

for (c, g), entries in group_entries.items():
    remaining = N_PER_LABEL - len(entries)
    if remaining <= 0:
        continue

    candidates = dev_by_group[(c, g)]
    rng.shuffle(candidates)

    # Greedily pick from unused videos first
    picked = []
    for cand in candidates:
        if len(picked) >= remaining:
            break
        if cand["video_id"] not in used_videos:
            picked.append(cand)
            used_videos.add(cand["video_id"])

    # If still need more, allow videos already used but prefer new frames
    if len(picked) < remaining:
        used_frame_keys = set(
            (e["frame"]["video_id"], e["frame"]["frame_id"])
            for grp in group_entries.values()
            for e in grp
        )
        used_frame_keys.update((p["video_id"], p["frame_id"]) for p in picked)
        for cand in candidates:
            if len(picked) >= remaining:
                break
            key = (cand["video_id"], cand["frame_id"])
            if key not in used_frame_keys:
                picked.append(cand)
                used_frame_keys.add(key)

    for p in picked:
        group_entries[(c, g)].append({
            "frame": p,
            "sample_group": {"criterion": c, "gt": g},
            "source": "train_dev",
        })

# Summary
total = 0
all_videos = set()
for (c, g), entries in group_entries.items():
    rest_n = sum(1 for e in entries if e["source"] == "train_rest")
    dev_n = sum(1 for e in entries if e["source"] == "train_dev")
    vids = set(e["frame"]["video_id"] for e in entries)
    all_videos.update(vids)
    total += len(entries)
    print(f"({c}, gt={g}): {len(entries)} total ({rest_n} rest + {dev_n} dev), {len(vids)} unique videos")

print(f"\nTotal: {total} frames from {len(all_videos)} unique videos")
print(f"  train_rest: {sum(1 for grp in group_entries.values() for e in grp if e['source']=='train_rest')}")
print(f"  train_dev:  {sum(1 for grp in group_entries.values() for e in grp if e['source']=='train_dev')}")

In [ ]:
# -----------------------------
# Build final_entries list
# -----------------------------
final_entries: List[Dict[str, Any]] = []
for (c, g) in group_specs:
    for entry in group_entries[(c, g)]:
        final_entries.append(entry)

print(f"Final batch size: {len(final_entries)} / 60")

# Show label combo distribution for train_rest picks
print("\nTrain_rest label combos:")
for entry in final_entries:
    if entry["source"] == "train_rest":
        f = entry["frame"]
        print(f"  group=({entry['sample_group']['criterion']},{entry['sample_group']['gt']}) "
              f"combo={f['label_combo']} video={f['video_id'][:12]}...")

In [ ]:
# -----------------------------
# Annotation UI
# -----------------------------
if not final_entries:
    raise RuntimeError("No images available for annotation.")

ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = (
    ANNOTATIONS_DIR
    / f"sages__rubrics_v3__seed{SEED}__batch{BATCH_IDX}__{run_ts}.jsonl"
)

CHOICES = ["yes", "no", "uncertain"]

progress_label = widgets.Label()
meta_out = widgets.Output()
image_out = widgets.Output()

item_widgets: Dict[str, widgets.RadioButtons] = {}
item_rows: List[widgets.Widget] = []
header_widgets: Dict[str, widgets.HTML] = {}

for criterion in ["C1", "C2", "C3"]:
    header = widgets.HTML("<h3></h3>")
    header_widgets[criterion] = header
    item_rows.append(header)
    for item in rubric_items_by_criterion[criterion]:
        item_id = item["id"]
        weight = item["weight"]
        text = item["text"]
        label_type = item.get("label_type", "")
        label = widgets.HTML(f"<b>{item_id}</b> (w={weight}, {label_type}): {text}")
        rb = widgets.RadioButtons(options=CHOICES, value="uncertain")
        item_widgets[item_id] = rb
        row = widgets.HBox([rb, label])
        item_rows.append(row)

notes_widget = widgets.Textarea(
    value="",
    placeholder="Optional notes...",
    description="Notes:",
    layout=widgets.Layout(width="100%", height="80px"),
)

save_button = widgets.Button(description="Save & Next", button_style="success")
skip_button = widgets.Button(description="Skip", button_style="warning")
reset_button = widgets.Button(description="Reset", button_style="")

status_out = widgets.Output()

state = {"index": 0}


def current_entry() -> Dict[str, Any]:
    return final_entries[state["index"]]


def format_gt_label(val) -> str:
    if val == 1:
        return "1"
    if val == 0:
        return "0"
    return "unknown"


def render_current() -> None:
    entry = current_entry()
    frame = entry["frame"]
    sample_group = entry["sample_group"]
    gt = frame["gt"]

    progress_label.value = (
        f"Image {state['index'] + 1} / {len(final_entries)}  "
        f"[{entry['source']}]"
    )

    header_widgets["C1"].value = f"<h3>C1 (GT={format_gt_label(gt.get('c1'))})</h3>"
    header_widgets["C2"].value = f"<h3>C2 (GT={format_gt_label(gt.get('c2'))})</h3>"
    header_widgets["C3"].value = f"<h3>C3 (GT={format_gt_label(gt.get('c3'))})</h3>"

    with meta_out:
        clear_output(wait=True)
        print("video_id:", frame["video_id"])
        print("frame_id:", frame["frame_id"])
        print("image_path:", frame["image_path"])
        print("source:", entry["source"])
        print("sample_group:", sample_group)
        print("gt:", gt)
        print("label_combo:", frame["label_combo"])

    with image_out:
        clear_output(wait=True)
        if SHOW_IMAGES:
            img_path = Path(frame["image_path"])
            title = f"{frame['video_id'][:12]}... | frame {frame['frame_id']} | {entry['source']}"
            show_image(img_path, title)


def reset_widgets() -> None:
    for rb in item_widgets.values():
        rb.value = "uncertain"
    notes_widget.value = ""


def build_record(*, skipped: bool) -> Dict[str, Any]:
    entry = current_entry()
    frame = entry["frame"]
    gt = frame["gt"]

    if skipped:
        rubrics = {item_id: "uncertain" for item_id in item_widgets}
    else:
        rubrics = {item_id: rb.value for item_id, rb in item_widgets.items()}

    record = {
        "dataset": "cvs_challenge_sages_v1",
        "source_manifest": entry["source"],
        "seed": SEED,
        "batch_idx": BATCH_IDX,
        "sample_group": entry["sample_group"],
        "image": {
            "image_path": frame["image_path"],
            "video_id": frame["video_id"],
            "frame_id": frame["frame_id"],
            "split": frame["split"],
        },
        "gt": gt,
        "label_combo": list(frame["label_combo"]),
        "rubric_version": "cvs_rubrics_v3",
        "rubrics": rubrics,
        "notes": notes_widget.value.strip(),
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

    if skipped:
        record["skip_reason"] = "skipped"

    return record


def append_record(record: Dict[str, Any]) -> None:
    with output_path.open("a") as f:
        f.write(json.dumps(record) + "\n")


def advance() -> None:
    if state["index"] >= len(final_entries) - 1:
        with status_out:
            clear_output(wait=True)
            print("Done. Reached end of batch.")
        save_button.disabled = True
        skip_button.disabled = True
        return
    state["index"] += 1
    reset_widgets()
    render_current()


def on_save(_):
    record = build_record(skipped=False)
    append_record(record)
    with status_out:
        clear_output(wait=True)
        print("Saved:", output_path.name)
    advance()


def on_skip(_):
    record = build_record(skipped=True)
    append_record(record)
    with status_out:
        clear_output(wait=True)
        print("Skipped:", output_path.name)
    advance()


def on_reset(_):
    reset_widgets()


save_button.on_click(on_save)
skip_button.on_click(on_skip)
reset_button.on_click(on_reset)

render_current()

controls = widgets.HBox([save_button, skip_button, reset_button, progress_label])

ui = widgets.VBox(
    [
        widgets.HTML(f"<b>Output:</b> {output_path}"),
        controls,
        meta_out,
        image_out,
        widgets.VBox(item_rows),
        notes_widget,
        status_out,
    ]
)

display(ui)

In [ ]:
# -----------------------------
# Optional: quick summary of saved annotations
# -----------------------------
if output_path.exists():
    rows = [json.loads(line) for line in output_path.read_text().splitlines() if line.strip()]
    print("Rows:", len(rows))
    if rows:
        item_ids = list(rows[0]["rubrics"].keys())
        uncertain_counts = {item_id: 0 for item_id in item_ids}
        for row in rows:
            for item_id, val in row["rubrics"].items():
                if val == "uncertain":
                    uncertain_counts[item_id] += 1
        top_uncertain = sorted(uncertain_counts.items(), key=lambda x: -x[1])[:5]
        print("Top uncertain items:")
        for item_id, count in top_uncertain:
            print(item_id, count)